# Cartographie des pays associés aux publications de Paris 8

Le notebook permet de cartographier les pays des auteurs qui ont signé une ou plusieurs publications avec des chercheur.es de Paris 8. Par Pays, on entend le pays des établissements auxquels ces auteurs sont associés.

Les données géographiques ont été récupérées à partir des infos contenues dans le Baromètre Science Ouverte de Paris 8 réalisé en 2025. Les scripts utilisés pour cette récupération sont disponibles dans le dossier `data_processing`. Nous utilisons en priorité le "ror" lorsque celui-ci est renseigné dans les données du BSO. S'il ne les pas, on utilise des outils de NLP pour reconnaître les entités nommées (spacy, regex). À partir de là on interroge l'api du Registry Organisation Research pour retrouver le ror.

Les données géographiques exploitées dans ce notebook ont été collectées à partir du ror en utilisant l'API. L'avantage est qu'on a ainsi des termes standardisés ainsi que des coordonnées géographique (lattitude et longitude)


In [39]:

import pandas as pd
import os
import s3fs

import datetime
import json
import re
import tqdm

In [40]:
#Library for social network analysis
import numpy as np
#import plotly.express as px


# Chargement des fichiers de données et préparation du dataframe

On charge le fichier créé à partur du notebook "network_hal_p8-Copy1.ipynb".

In [41]:
# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"




In [42]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_ror_with_geographic_info.csv", "r") as file_in:
    df_inst = pd.read_csv(file_in, sep=",")


# Le fichier contenant les publications, les auteurs et leur affiliations
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-20_affiliation_with_info.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep=",")

#df0.drop(columns=["Unnamed: 0"], inplace=True)

PermissionError: Forbidden

In [43]:
df0

,id,full_name,row_id,rors,ror,nom,country_name,city,geonames_id
0,halhal-03933378,Philippe Blanchard,halhal-03933378_2,no ror|https://ror.org/027mnq498|https://ror.o...,https://ror.org/027mnq498,"De la Préhistoire à l'Actuel : Culture, Enviro...",France,Pessac,2987805
1,halhal-03933378,Philippe Blanchard,halhal-03933378_2,no ror|https://ror.org/027mnq498|https://ror.o...,https://ror.org/057qpr032,Université de Bordeaux,France,Bordeaux,3031582
2,halhal-03933378,Philippe Blanchard,halhal-03933378_2,no ror|https://ror.org/027mnq498|https://ror.o...,https://ror.org/02feahw73,Centre National de la Recherche Scientifique,France,Paris,2988507
3,halhal-03933378,Jean-Philippe Chimier,halhal-03933378_3,no ror|https://ror.org/04ahza989|https://ror.o...,https://ror.org/04ahza989,"CItés, Territoires, Environnement et Sociétés",France,Tours,2972191
4,halhal-03933378,Jean-Philippe Chimier,halhal-03933378_3,no ror|https://ror.org/04ahza989|https://ror.o...,https://ror.org/02wwzvj46,Université de Tours,France,Tours,2972191
...,...,...,...,...,...,...,...,...,...
409016,halhal-02120667,Guillaume Bruniaux,halhal-02120667_124113,https://ror.org/00r8amq78|https://ror.org/04mv...,https://ror.org/02feahw73,Centre National de la Recherche Scientifique,France,Paris,2988507
409017,halhal-01964707,Coralie Vincent,halhal-01964707_124114,https://ror.org/00nrven85|https://ror.org/04we...,https://ror.org/00nrven85,Structures Formelles du Langage,France,Saint-Denis,2980916
409018,halhal-01964707,Coralie Vincent,halhal-01964707_124114,https://ror.org/00nrven85|https://ror.org/04we...,https://ror.org/04wez5e68,Université Paris 8,France,Paris,2988507
409019,halhal-01964707,Coralie Vincent,halhal-01964707_124114,https://ror.org/00nrven85|https://ror.org/04we...,https://ror.org/02feahw73,Centre National de la Recherche Scientifique,France,Paris,2988507


## Regroupement par pays

Pour chaque pays, on compte le nombre de publications associés via la fonction "groupby" de pandas. 

In [ ]:
df_country = df0[["id","country_name",]].drop_duplicates()
df_country

In [ ]:
freq_country = df_country.groupby(["country_name"]).agg(nb=("id","size")).sort_values("nb", ascending= False).reset_index()
freq_country["freq"] = freq_country.nb/ np.sum(freq_country.nb)
freq_country.head(20)

On constate ainsi que 95% des publications ont au moins un auteur appartenant à un organisme de recherche français. Ce qui n'est pas surpprenant puisque c'est la condition pour qu'une publication figure dans le BSO

Il peut donc être intéressant de retirer ces deux modalités du décomptes et, *in fine*, de la représentation cartographique.


In [ ]:
freq_country = df_country.loc[df_country.country_name!="France"].groupby(["country_name"]).agg(nb=("id","size")).sort_values("nb", ascending= False).reset_index()
freq_country["freq"] = freq_country.nb/ np.sum(freq_country.nb)*100
freq_country.head(10)

# Regroupement par villes

## Villes françaises

In [ ]:
df_city = df0[["id","city","country_name"]].drop_duplicates()
french_city = df_city.loc[df_city.country_name=="France"].groupby(["city"]).agg(nb=("id","size")).sort_values("nb", ascending= False).reset_index()
french_city["freq"] = french_city.nb/ np.sum(french_city.nb)*100
french_city.head(10)

# Villes étrangères

In [ ]:

other_city = df_city.loc[df_city.country_name!="France"].groupby(["city"]).agg(nb=("id","size")).sort_values("nb", ascending= False).reset_index()
other_city["freq"] = other_city.nb/ np.sum(other_city.nb)*100
other_city.head(10)

## Organisations

In [ ]:
df_org = df0[["id","nom","country_name"]].drop_duplicates()
org = df_org.loc[df_org.country_name=="France"].groupby(["nom"]).agg(nb=("id","size")).sort_values("nb", ascending= False).reset_index()
org["freq"] = org.nb/ np.sum(org.nb)*100
org.head(10)

In [ ]:
org1 = df_org.loc[df_org.country_name!="France"].groupby(["nom"]).agg(nb=("id","size")).sort_values("nb", ascending= False).reset_index()
org1["freq"] = org1.nb/ np.sum(org1.nb)*100
org1.head(10)

In [ ]:
import geopandas as gp
import geopy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import contextily as ctx
import matplotlib.pyplot as plt
import plotly.express as px #if using plotly

## Retrouver les coordonnées géospaciale des pays

Nous utilisons les modules geopandas et geopy qui permettent de manipuler facilement les données géospaciales. Geopandas est d'une certaine manière une extension de pandas.
Geopy propose des fonctions pour retrouver les coordonnées d'entité géographique (villes, régions, pays, etc.

**EPSG** est l'acronyme d'European Petroleum Survey Group. Code associé à l'un des paramètres géodésiques établis par l'EPSG et gérés par le Comité sur la géomatique de l'IOGP 

In [ ]:

dfg = gp.GeoDataFrame(other_city)


# Source - https://stackoverflow.com/a/61346493
# Posted by hyances, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-18, License - CC BY-SA 4.0

dfg['geometry'] = gp.tools.geocode(dfg.city).geometry 
dfg.crs = "EPSG:4326" 
dfg.head()



On a une nouvelle colonne ("geometry") qui correspond à la longitude et la latitude : `POINT (LONG LAT)`. On enregistre le dataframe pour pouvoir le réutiliser plus tard


In [ ]:
FILE_PATH_OUT_S3 = BUCKET_OUT + "/Data_bso/outputs/publis/geographical_analyse/2026-02-19_country_name_with_location_point"
with fs.open(f"{FILE_PATH_OUT_S3}.csv", 'w') as file_out:
    dfg.to_csv(file_out, sep = ",", index = False)
    


In [ ]:
dfg1 = dfg.to_crs(epsg=3857)
ax = dfg1.plot(figsize=(16, 10), alpha=0.75, edgecolor='k', marker='o', color='red', markersize=dfg1.freq*100)
ctx.add_basemap(ax)
ax.set_axis_off()

In [ ]:
df_inst.columns

In [ ]:
city_coordinate = df_inst[["city", "country_name", "lat", "lng"]].drop_duplicates()
other_city = other_city.merge(city_coordinate, on = ["city"], how = "left")
other_city

In [ ]:
import plotly.graph_objects as go

import pandas as pd

other_city['text'] = other_city['city'] + '<br>Number of publication ' + other_city['nb'].astype(str)
limits = [(0,3),(3,11),(11,21),(21,51),(51,101),(101,201)]
colors = ["royalblue","crimson","lightseagreen","orange","pink","yellow"]
cities = []
scale = 5000

fig = go.Figure()

for i in range(len(limits)):
    lim = limits[i]
    df_sub = other_city.loc[(other_city.nb>=lim[0]) & (other_city.nb<lim[1])]
    fig.add_trace(go.Scattergeo(
        locationmode = 'ISO-3',
        lon = df_sub['lng'],
        lat = df_sub['lat'],
        text = df_sub['text'],
        marker = dict(
            size = df_sub['nb'],
            color = colors[i],
            line_color='rgb(40,40,40)',
            line_width=0.5,
            sizemode = 'area'
        ),
        name = '{0} - {1}'.format(lim[0],lim[1])))

fig.update_layout(
        title_text = 'Villes des auteurs ayant publié avec un chercheur de Paris 8<br>(Click legend to toggle traces)',
        showlegend = True,
        geo = dict(
            scope = 'world',
            landcolor = 'rgb(217, 217, 217)',
        )
    )

fig.show()
fig.write_html("../figures/maps/map_of_cities_2026-03-24_BSO_Paris8.html")

Comme on le voit sur la carte, la fréquence des publications est représentée par un point. Cela est lié au fait que dans la colonne "geometry" on utilise la longitude et la latitude. Dans les cellules ci-dessous, nous allons récupérer cette fois les coordonnées correspondant aux "frontières" des pays. De cette manière, nous pourrons colorier la surface des pays en fonction de la fréquence des coopérations (mesurée en nombre de publication), c'est-à-dire obtenir ce qu'on appelle une corothplèthe.

* un exemple de corothplèthe:

![https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Carte_figurative_de_l%27instruction_populaire_de_la_France.jpg/330px-Carte_figurative_de_l%27instruction_populaire_de_la_France.jpg](https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Carte_figurative_de_l%27instruction_populaire_de_la_France.jpg/330px-Carte_figurative_de_l%27instruction_populaire_de_la_France.jpg)



Pour cela, il nous faut d'abord récupérer les coordonées des frontières de chaque pays à partir des données disponibles sur le web. Nous proposons ici d'utiliser les jeux de données disponible sur l'entrepôt recherche.data.gouv de la [*Toulouse School of Economics*](https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi:10.57745/ABJ8OQ) :  

> James, Kyllian, 2025, "World Countries Boundaries", [https://doi.org/10.57745/ABJ8OQ](https://doi.org/10.57745/ABJ8OQ), Recherche Data Gouv, V1

Il contient des pays tels que Taiwan qui ne sont pas dans les [fichiers proposés par la Banque Mondiale](https://datacatalog.worldbank.org/search/dataset/0038272/world-bank-official-boundaries).

In [ ]:

FILE_PATH_OUT_S3 = BUCKET_OUT + "/Data_bso/referentiels/World_Countries_Boundaries.geojson"
with fs.open(FILE_PATH_OUT_S3, 'r') as file_in:
    #dfg.to_csv(file_out, sep = ",", index = False)
    map_df = gp.read_file(file_in)



In [ ]:
map_df.crs = "EPSG:4326"


On peut facilement visualiser les données obtenues avec la fonction `gp_dataframe.plot()`

In [ ]:
map_df.plot()

## "Fusion" des dataframe dfg et map_df

Une fois les données des frontières obtenues, nous pouvons fusionnés le dataframe avec celui que nous avons créés précédemment à partir des pays "co-publiant" avec les chercheur.es de Paris 8. Toutefois, ke nom des pays peut varier. Nous allons alors nous appuyés sur la longitude et la latitude. Nous récupérons les coordonnées à l'aide `gp.tools.geocode`.

In [ ]:
#GET POint of country

map_df['geometry'] = gp.tools.geocode(map_df.COUNTRY).geometry 
map_df.crs = "EPSG:4326"
map_df.head(5)
    

On ne garde que les colonnes utiles et nous fusionnons les deux dataframe en utilisant la colonne geometry pour faire la jointure.


In [ ]:
#Merge dfg with map_df
list_of_column = ['NAME', 'VISUALIZATION_NAME', 'ISO2','SOVRN','COUNTRY', 'CONTINENT','SOURCE', 'geometry']
dfg_harm = dfg.merge(map_df[list_of_column], on = ["geometry"], how = "left")
dfg_harm.loc[dfg_harm.COUNTRY.isna()]

On constate malheureusement que dans certains cas, la jointure n'a pas fonctionné. Les coodonnées ne correspondent pas.

Comme les données manquantes ne sont pas nombreuses, nous allons les complétées manuellement en allant chercher les coordonées dans `map_df`.

In [ ]:
map_df.loc[map_df.VISUALIZATION_NAME.str.contains("Russia")]
unmatch

In [ ]:
unmatch = dfg_harm.loc[dfg_harm.COUNTRY.isna()]

# dictionnaire faisant correspond le name_country de dfg avec 'VISUALIZATION_NAME' dans map_df
dict_state = {'United States' : 'United States of America',
              'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland',
              'South Korea':"Republic of Korea",
              'Taiwan': 'Taiwan (Province of China)',
              'The Netherlands':'Netherlands',
              'Russia':'Russian Federation'
              }

# On crée une nouvelle colonne 'VISUALIZATION_NAME' et on merge unmatch avec map_df en prenant 'VISUALIZATION_NAME' comme colonne de jointure
unmatch["VISUALIZATION_NAME"] = unmatch.country_name.map(dict_state.get)
unmatch = unmatch[["country_name","nb","freq","VISUALIZATION_NAME"]].merge(map_df[list_of_column], on = ['VISUALIZATION_NAME'], how = "left")

unmatch

Nous n'avons plus de données manquantes. Nous pouvons reconstituer le jeu de données en concaténant les lignes qui ne posaient pas de problème avec les 5 lignes que nous venons de compléter.

In [ ]:
match = dfg_harm.loc[~dfg_harm.COUNTRY.isna()]


In [ ]:
dfg_harm = pd.concat([match,unmatch])
dfg_harm

Les noms de pays étant maintenant harmonisés, nous pouvons joindre les coordonnées des frontières au dataframe 'dfg_harm' et tracer enfin notre corothplèthe.

In [ ]:
FILE_PATH_OUT_S3 = BUCKET_OUT + "/Data_bso/referentiels/World_Countries_Boundaries.geojson"
with fs.open(FILE_PATH_OUT_S3, 'r') as file_in:
    #dfg.to_csv(file_out, sep = ",", index = False)
    map_poly = gp.read_file(file_in)
map_poly.crs = "EPSG:4326"
dict_polygone = dict(zip(map_poly.VISUALIZATION_NAME, map_poly.geometry))

dfg_harm = dfg_harm.rename(columns={"geometry":"point"})
dfg_harm["geometry"] = dfg_harm.VISUALIZATION_NAME.map(dict_polygone.get)


In [ ]:
dfg_harm.drop(columns=["point"], inplace=True)

In [ ]:
dfg_harm.to_file("2026-03-20_country_name_with_polygone.geojson", driver ="GeoJSON")

In [ ]:
dfg_harm = gp.read_file("2026-03-20_country_name_with_polygone.geojson")
dfg_harm.crs = "EPSG:4326"



## Vue mondiale

In [ ]:
fig, ax = plt.subplots(figsize = (20,20))
map_poly.plot(ax=ax, color='lightgrey')
dfg_harm.plot(column="freq", cmap='Blues', linewidth=0.8, ax=ax, edgecolor='0.8')
ax.set_title('Pays des auteurs ayant signé une ou plusieurs publications avec des chercheur.es de Paris 8 (données issues du Barometre science ouverte)')

## Zoom sur l'europe

In [ ]:
dfg_europe = dfg_harm.loc[(dfg_harm.CONTINENT == "EUROPE") & (dfg_harm.VISUALIZATION_NAME!='Russian Federation')]
map_europe = map_poly.loc[(map_poly.CONTINENT == "EUROPE") & (map_poly.VISUALIZATION_NAME!='Russian Federation')]



In [ ]:
fig, ax = plt.subplots(figsize = (20,20))
map_europe.plot(ax=ax, color='lightgrey',linewidth=0.8, edgecolor='0')
dfg_europe.plot(column="freq", cmap='Blues', linewidth=0.8, ax=ax, edgecolor='0')
ax.set_title('Pays des auteurs ayant signé une ou plusieurs publications avec des chercheur.es de Paris 8 (données issues du Barometre science ouverte)')

In [ ]:
dfg_africa = dfg_harm.loc[(dfg_harm.CONTINENT == "AFRICA")]
map_africa = map_poly.loc[(map_poly.CONTINENT == "AFRICA")]



In [ ]:
fig, ax = plt.subplots(figsize = (20,20))
map_africa.plot(ax=ax, color='lightgrey',linewidth=0.8, edgecolor='0')
dfg_africa.plot(column="freq", cmap='Blues', linewidth=0.8, ax=ax, edgecolor='0')
ax.set_title('Pays des auteurs ayant signé une ou plusieurs publications avec des chercheur.es de Paris 8 (données issues du Barometre science ouverte)')

In [ ]:

fig = px.choropleth(dfg_europe, 
                    geojson=dfg_europe.geometry,
                    locations=dfg_europe.index,
                    color="nb",
                    color_continuous_scale="Viridis"
)
fig.show()

In [ ]:
# Import the data from the web
df = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/fips-unemp-16.csv",
                   dtype={"fips": str})

                   # Check the distribution of the variable with seaborn:
import seaborn as sns
sns.set_theme(style="darkgrid")
sns.histplot(data=df, x="unemp")
plt.show();



In [ ]:
# Load the county boundary coordinates
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)


# Build the choropleth
import plotly.express as px
fig = px.choropleth(df, 
    geojson=counties, 
    locations='fips', 
    color='unemp',
    color_continuous_scale="Viridis",
    range_color=(0, 12),
    scope="usa",
    labels={'unemp':'unemployment rate'}
)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

# Improve the legend
fig.update_layout(coloraxis_colorbar=dict(
    thicknessmode="pixels", thickness=10,
    lenmode="pixels", len=150,
    yanchor="top", y=0.8,
    ticks="outside", ticksuffix=" %",
    dtick=5
))

fig.show()

## Identification des adresses des villes